# RNN, Recurrent neural network

$x_t$: 第t个时刻的输入;

$y_t$: 第t个时刻的输出；

$h_t$: 第t个时刻更新后的记忆，即隐状态；

$w_h$: 隐藏层的权重；b_h: 隐藏层的偏置；

$w_y$: 输出层的权重；b_y: 输出层的偏置。

当前时刻的隐状态等于前一时刻的隐状态和当前时刻的序列输入进行拼接，然后经过隐藏层线性变化和激活函数后的输出：
$$h_t = tanh([h_{t-1} | x_t]w_h + b_h)$$

输出层是一个多分类，激活函数用softmax：
$$y_t = Softmax(h_tw_y + b_y)$$

---

- 循环层：RNN的循环实际发生在第一层，即拼接后计算输出$h_t$时，参数是共享的，被多次更新；
- 普通层：不参与循环，梯度更新只发生在这一步，是独立的，参数只会在当前更新一次，只是为了完成每一步的特定任务
- 优势：通过隐状态机制赋予模型记忆能力，利用历史信息；循环层参数共享，提高效率和泛化能力；能处理任意长度的输入序列

---

- 多对一：输入序列长度有多个token，输出只有一个用来进行分类的向量，如情感分类。除了最后一个时间步有输出外，其他时间步都只更新隐藏状态；
- 一对多：输入一个向量，然后生成输出token序列，如小说生成。第一个时间步输入$x$，输出第一个token: $y_1$，然后将$y_1$作为第二个时间步的输入token，继续生成输出$y_2$，……，以此类推，直到模型生成结束符 \<EOS>
- 多对多：
    - 长度一致：输入序列和输出序列长度一致，比如NER任务（命名实体识别,named entity recognition）。每个时间步的输入都是之前隐藏状态和当前时间步的输入，输出分为当前时间步的输出以及当前RNN的隐藏状态；
    - 长度不一致：输入序列和输出序列长度不一致，比如翻译任务。第一个阶段编码，每个时间步不进行输出，只更新隐藏状态；当读入了所有 x 之后，进入解码阶段，每个时间步会输出一个目标token，同时将输出的token作为下一时间步的输入token，直到输出序列结束符 \<EOS>
- 一对一：只有一个时间步，退化成一个全连接神经网络

---

RNN的问题：

- 循环层传递时，梯度消失/爆炸。RNN实际上是在时间维度上做深层网络，梯度同样不稳定；
- 长期依赖问题，很难让 $x_1$ 影响到 y，信息每一步都被压缩、损耗，隐状态本身传递的是一种短期记忆 short-term memory；
- 不能并行计算，$h_t$ 依赖 $h_{t-1}$，GPU利用率低

# LSTM, Long Short-Term Memory

- *新信息经过输入门筛选*：当有一个新信息 Z，是一个神经网络的输出logits值，是一个包含多维度的向量。经过一个**门函数**控制哪些维度可以写入到长期记忆里。这个门函数是$Sigmoid$，因为其值接近0或1，所以与Z按位相乘可以看作是允许某些维度的新信息进入长期记忆、某些维度的新信息不能进入长期记忆、某些维度的信息可以部分进入长期记忆，这个门叫做**输入门**；
- *输入门筛选后的新信息进入长期记忆*：
    - 取出记忆细胞内的长期记忆，通过一个 Sigmoid 的**遗忘门**，来遗忘长期记忆里某些维度的信息；
    - 把经过筛选的新信息和经过遗忘的长期记忆按位相加，然后经过 tanh 激活，作为当前记忆细胞的待输出记忆；
- *输出*：
    - 激活后的待输出记忆，还要经过一个 sigmoid 的**输出门**来控制哪些值适合在当前时间步输出（输出对应的是RNN的隐状态）；
    - 如果每个时间步需要输出，就对上面的隐状态输出增加一个普通层即可

\* 原来的RNN在多时间步之间传递一个隐状态 h，现在就再多传递一个记忆细胞状态 c；

\* 新信息向量$Z$,输入门输入向量$Z_i$,遗忘门输入向量$Z_f$,输出门输入向量$Z_o$，实际上都是上一个状态$h_{t-1}$和当前时刻的$x_t$拼接后作为输入，分别做4个线性回归得到的logits值，这4个线性回归的权重分别是$w_h, w_i, w_f, w_o$持续更新；

\* 记忆细胞的长期记忆有一个直接连接的通道，这条通道上没有线性回归，没有激活函数。长期记忆容易保持，反向传递时，梯度也更容易传递到前边的时间步。

---

新信息的logits：

$Z = [h_{t-1} | x_t]w_h+b_h$

输入门：

$Z_i = [h_{t-1} | x_t]w_i+b_i$

$G_i = Sigmoid(Z_i)$

遗忘门：

$Z_f = [h_{t-1} | x_t]w_f+b_f$

$G_f = Sigmoid(Z_f)$

输出门：

$Z_o = [h_{t-1} | x_t]w_o+b_o$

$G_o = Sigmoid(Z_o)$

记忆细胞状态：

$c_t = c_{t-1} \odot G_f + tanh(Z) \odot G_i$

隐状态：

$h_t = tanh(c_t) \odot G_o$

# GRU, Gated Recurrent Unit

对 lstm 的简化，去掉了记忆细胞状态$c_t$，而仅用隐状态$h_t$就解决了长期记忆和梯度消失的问题

- 通过**重置门**$G_r$决定从长期记忆里去掉一些信息。用 Sigmoid 函数，输入是上一时刻的记忆$h_{t-1}$和当前时刻的输入$x_t$拼接后经过一个线性层得到的，线性层权重为$w_r$；
- 重置后的长期记忆和当前输入$x_t$合并，然后经过一个线性层（权重为$w_h$），加 tanh 激活，就得到**备用输出**$\tilde{h_t}$；
- 由于GRU只靠隐状态传递长期记忆，所以需要一个**更新门**来同时决定保留多少长期记忆、更新多少当前步产生的记忆。更新门生成的更新向量与备用输出按位相乘，得到要更新到长期记忆里的信息；用1减去更新向量里的每一维，得到对长期记忆的保留向量，按位相乘得到保留的长期记忆；相加得到这一步输出的长期记忆$h_t$

---

重置门：

$G_r = sigmoid([h_{t-1} | x_t]w_r + b_r)$

更新门：

$G_u = sigmoid([h_{t-1} | x_t]w_u + b_u)$

备用输出：

$\tilde{h_t} = tanh([h_{t-1} \odot G_r | x_t]w_h + b_h)$

隐状态：

$h_t = G_u \odot \tilde{h_t} + (1-G_u) \odot h_{t-1}$

# BiRNN, bidirectional recurrent neural network

- 可以正向和反向循环的双向循环神经网络；
- 单向循环的局限：只有前边token的隐状态，如果需要依赖后续的token就不能判断；
- BiRNN的结构：创建两个RNN(LSTM/GRU同理)，一个对序列正向循环，一个对序列反向循环，然后把某个序列元素在正向和反向循环的对应输出的隐状态合并，再作为完成特定任务的普通层的输入，此时就同时包含该元素前边和后边的信息；
- 局限：必须输入完整的序列进行处理，不适合生成类的任务

# Attention

- 在Seq2Seq模型中需要用到 encoder-decoder 的模式，所有信息在encoder中被压缩成了一个最终输出的隐状态$h_t$，不可避免的损失了长句中的信息；
- 引入attention机制，让Decoder在生成内容时，可以把注意力放在Encoder不同时间步的隐状态上作为参考信息，其输入由以下部分组成：
    - 上一时刻的隐状态（LSMT还有上一时刻的细胞状态）；
    - 上一时刻输出token的embedding（上一时刻的输出就是这一时刻的输入）；
    - 通过注意力观察后的由Encoder不同时间步的隐状态构成的注意力向量（即使是LSTM，也只用隐状态。因为隐状态更能代表当时时间步的信息，细胞状态是长期记忆信息）。
- 注意力机制实际就是将encoder在每个时间步输出的隐状态的值进行加权求和，生成一个注意力向量：
$$attention = \sum^t_{i=1} \alpha_i h_i,$$
其中，$\alpha_i$是标量，表示在翻译当前token时对$h_i$的注意力，且$\sum^t_{i=1} \alpha_i=1$。$h_i$是encoder在第 i 个时间步输出的隐状态；
- 计算注意力权重$\alpha_i$：
    - encoder输入token为$x_i$，输出的隐状态$h_i$。最后一个隐状态$h_T$作为decoder的隐状态输入，用s表示decoder的隐状态，所以$h_T=s_0$。decoder每一步生成的token作为下一步的输入，并且产生中间隐状态$s_i$
    - 首先，将$s_0$和$h_i$分别拼接，得到输入向量，然后分别经过全连接神经网络，得到标量的logits值，softmax之后即得到$h_i$对应的$\alpha_i$权重，加权之后得到Decoder第一个时间步的注意力向量$att_0=\sum^t_{i=1} \alpha_i h_i$。$[y_o | att_0]$输入RNN后得到隐状态$s_1$输出；
    - decoder的第二个时间步同上，将$s_1$和$h_i$分别拼接，得到输入向量，然后分别经过全连接神经网络得到logits值，softmax之后得到第二个时间步对$h_i$的$\alpha_i$权重

翻译模型、数据准备

In [ ]:
# 构建词典
import sentencepiece as spm

spm.SentencePieceTrainer.Train(
    input="D:/agent/torch/data/en2cn/train_en.txt",
    model_prefix="C:/Users/61075/PycharmProjects/learning/torch/runs/tokenizer/en_bpe",
    vocab_size=16000,
    model_type="bpe",
    character_coverage=1.0,
    unk_id=0,
    pad_id=1,
    bos_id=2,
    eos_id=3
)

spm.SentencePieceTrainer.Train(
    input="D:/agent/torch/data/en2cn/train_zh.txt",
    model_prefix="C:/Users/61075/PycharmProjects/learning/torch/runs/tokenizer/zh_bpe",
    vocab_size=16000,
    model_type="bpe",
    character_coverage=0.9995,
    unk_id=0,
    pad_id=1,
    bos_id=2,
    eos_id=3
)

In [1]:
# 使用训练的词典模型进行分词
import sentencepiece as spm

sp_cn = spm.SentencePieceProcessor()
sp_cn.load(r"C:\Users\61075\PycharmProjects\learning\torch\runs\tokenizer\zh_bpe.model")
sp_en = spm.SentencePieceProcessor()
sp_en.load(r"C:\Users\61075\PycharmProjects\learning\torch\runs\tokenizer\en_bpe.model")

def tokenize_en(text):
    return sp_en.encode(text, out_type=int)
def tokenize_cn(text):
    return sp_cn.encode(text, out_type=int)

PAD_ID = sp_en.pad_id()  # 1
UNK_ID = sp_en.unk_id()  # 0
BOS_ID = sp_en.bos_id()  # 2
EOS_ID = sp_en.eos_id()  # 3


text = '今天天气非常好。'

encode_result = sp_cn.encode(text, out_type=int)
print("encode: ", encode_result)

decode_result = sp_cn.decode(encode_result)
print("decode: ", decode_result)

encode:  [387, 3205, 5241, 11821]
decode:  今天天气非常好。


In [2]:
# 定义数据集
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

class TranslationDataset(Dataset):
    ## 初始化方法，读取英文和中文训练文本
    ## 给每个句子前后增加<bos>和<eos>
    ## 防止训练时显存不足，对长度超过限制的句子进行过滤

    def __init__(self, src_file, trg_file, src_tokenizer, trg_tokenizer, max_len=100):
        with open(src_file, "r", encoding="utf-8") as f:
            src_lines = f.read().splitlines()
        with open(trg_file, "r", encoding="utf-8") as f:
            trg_lines = f.read().splitlines()
        assert len(src_lines) == len(trg_lines)
        self.pairs = []
        self.src_tokenizer = src_tokenizer
        self.trg_tokenizer = trg_tokenizer

        for src, trg in zip(src_lines, trg_lines):
            # 每个句子前边增加<bos>后边增加<eos>
            src_ids = [BOS_ID] + self.src_tokenizer(src) + [EOS_ID]
            trg_ids = [BOS_ID] + self.trg_tokenizer(trg) + [EOS_ID]
            # 只保留输入和输出序列token数同时小于max_len的训练样本
            if len(src_ids) <= max_len and len(trg_ids) <= max_len:
                self.pairs.append((src_ids, trg_ids))   # 直接保存token id序列

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_ids, trg_ids = self.pairs[idx]
        return torch.LongTensor(src_ids), torch.LongTensor(trg_ids)

    ## 对一个batch的输入和输出token，依照最长的序列长度，用<pad>填充，确保一个batch的数据形状一致，组成一个tensor
    @ staticmethod
    def collate_fn(batch):
        src_batch, trg_batch = zip(*batch)
        src_lens = [len(src) for src in src_batch]
        trg_lens = [len(trg) for trg in trg_batch]
        src_pad = nn.utils.rnn.pad_sequence(src_batch, padding_value=PAD_ID)
        trg_pad = nn.utils.rnn.pad_sequence(trg_batch, padding_value=PAD_ID)
        return src_pad, trg_pad, src_lens, trg_lens


In [5]:
# 数据集太大了，只取一部分
input_file = r"D:\agent\torch\data\en2cn\train_zh.txt"
output_file = r"D:\agent\torch\data\en2cn\train_zh_10000.txt"
with open(input_file, "r", encoding="utf-8") as fin, \
     open(output_file, "w", encoding="utf-8") as fout:

    for i, line in enumerate(fin):
        if i >= 10000:
            break
        fout.write(line)

In [3]:
# 读取数据
dataset = TranslationDataset(
    src_file=r"D:\agent\torch\data\en2cn\train_en_10000.txt",
    trg_file=r"D:\agent\torch\data\en2cn\train_zh_10000.txt",
    src_tokenizer=tokenize_en,
    trg_tokenizer=tokenize_cn,
)
loader = DataLoader(dataset, batch_size=4, collate_fn=TranslationDataset.collate_fn)
for src, trg, _, _ in loader:
    print(src.shape, trg.shape)     # shape: [seq_len, batch_size]
    print(src, trg)
    break

torch.Size([24, 4]) torch.Size([21, 4])
tensor([[    2,     2,     2,     2],
        [   76,    76,    76,    76],
        [ 4932,  4932,  1534,  4932],
        [   42,    42,    42,    42],
        [ 1140,   494,  2305,    45],
        [  143,  1473,  3200,  1244],
        [  494,   291,   661, 15880],
        [ 4442,   450,    43,    98],
        [  494,    28,  3837,  5620],
        [ 7301,  7719, 15878,    43],
        [  117,    66, 15860,   115],
        [   83,   353,   514,     6],
        [ 3282,  5001, 15869,  4932],
        [  172,   132,     3,    42],
        [  347,   273,     1,    45],
        [ 7719,   237,     1,  1244],
        [   22,   291,     1, 15880],
        [ 6445,   450,     1,     3],
        [    3,    84,     1,     1],
        [    1,   189,     1,     1],
        [    1,  4104,     1,     1],
        [    1, 15856,     1,     1],
        [    1, 15869,     1,     1],
        [    1,     3,     1,     1]]) tensor([[    2,     2,     2,     2],
        [

BLEU评价指标：

- Bilingual Evaluation Understudy，衡量翻译结果和一个或多个参考译文之间的相似度；
- 原理：
    - 将模型翻译句子和参考译文都转化为token序列，
    - 计算每个模型翻译的token在参考译文的token序列中出现率$p_1$，
    - 计算每两个连续翻译token序列在参考token序列中出现率$p_2$，
    - 计算每三个、每四个连续翻译token序列在参考token序列中出现率$p_3，p_4$，
    - 综合$p_1, p_2, p_3, p_4$，并惩罚模型输出翻译太短的情况，得到最终BLEU得分

---

- $p_1$：衡量模型翻译$T_m$单个token参照参考翻译$T_r$的准确率
    - count：某个token在$T_m$中出现的个数；
    - clip_count：某个token在$T_m$和$T_r$中出现较少的那个count值；
    - $p_1 = \frac{\sum clip\_count}{\sum count}$
- $p_2$：衡量模型翻译$T_m$token序列中两个连续token参照参考翻译$T_r$的准确率
- $p_3$、$p_4$同理
- 对p值求几何平均：$\prod^4_{n=1}p_n^{\frac{1}{4}}=exp(\sum^4_{n=1}\frac{1}{4}\log{p_n})$
- 短句惩罚 brevity penalty：在p值几何平均数的结果上乘一个值
$$
BP = \begin{cases}
1 & m > r \\
exp(1-\frac{r}{m}) & m \le r
\end{cases}
$$
\* 其中，r是参考翻译的长度，m是模型翻译的长度
- 最终：$$BLEU = BP \times exp(\sum^4_{n=1}\frac{1}{4}\log{p_n})$$
或单独为每个p值设置不同的权重$w_n$
$$BLEU = BP \times exp(\sum^4_{n=1}w_n\log{p_n})$$

调用BLEU

In [11]:
import sacrebleu

# 单个参考译文列表
references = [
    "今天天气很好。",
    "我喜欢在雨天散步。",
    "今天要早点下班。"
]

# 模型生成的候选翻译
hypotheses = [
    "今日天气不错。",
    "下雨的时候我喜欢散步。",
    "今天下班要早些。"
]

# sacrebleu 需要 references 是 list[list[str]]
# 即使只有一个参考，也要是内层 list
references = [references]   # 变成：[[ref1, ref2, ref3]]

# 计算bleu
bleu = sacrebleu.corpus_bleu(hypotheses, references, tokenize='zh')
print(f"BLEU score: {bleu.score:.2f}%")  # 0.3以上认为基本能用了

BLEU score: 16.31%


定义翻译模型

In [4]:
# encoder

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hid_dim, n_layers=3):
        super(Encoder, self).__init__()
        # 默认3个循环层
        self.n_layers = n_layers

        # 定义Embedding，将BPE分词输出的token_ids转化为emb_dim的embedding向量
        # <pad>不参与运算，它的embedding不用学习
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)

        # 定义双向LSTM模型
        self.bi_lstm = nn.LSTM(embed_dim, hid_dim, n_layers, bidirectional=True)

        # 定义线性层列表来降低维度，因为encoder是双向的，隐状态和记忆细胞维度是hid_dim*2，
        # 而decoder是单向的，隐状态和记忆细胞维度是hid_dim
        self.fc_hidden = nn.ModuleList([
            nn.Linear(2 * hid_dim, hid_dim) for _ in range(n_layers)
        ])
        self.fc_cell = nn.ModuleList([
            nn.Linear(2 * hid_dim, hid_dim) for _ in range(n_layers)
        ])

    def forward(self, src, src_lens):
        embedded = self.embedding(src)

        # 将一个 padded sequence(已经填充到统一长度的batch序列)转换为一个特殊的 PackedSequence 对象
        # 在传入 RNN 时能跳过padding部分的计算
        packed = nn.utils.rnn.pack_padded_sequence(embedded, src_lens, enforce_sorted=False)

        # outputs：形状为[seq_len, batch_size, hid_dim*2]，表示每个时间步、最后一层LSTM的双向隐状态拼接；
        # (hidden, cell)：形状都是[num_layers*2, batch_size, hid_dim]，表示每一层、每个方向在最后一个时间步上的隐状态或记忆细胞状态
        # outputs是每个时间步的，用于计算attention；hidden是最后时间步的，用于decoder初始化输入
        outputs, (hidden, cell) = self.bi_lstm(packed)

        # PackedSequence 类型的输出还原成带 padding 的标准tensor
        outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs)  # [src_len, batch, hid_dim*2]

        # 重塑隐状态和记忆细胞状态，让 层 和 方向 显式分离
        # [n_layers*2, batch, hid_dim] -> [n_layers, 2, batch, hid_dim]
        # 其实就是将第0维拆成layer * direction两个维度
        hidden = hidden.view(self.n_layers, 2, -1, hidden.size(2))
        cell = cell.view(self.n_layers, 2, -1, cell.size(2))

        # 为每一层处理前向和后向状态
        final_hidden = []
        final_cell = []
        for layer in range(self.n_layers):
            # 对每一层，将正向和反向的隐状态、细胞状态合并，通过一个线性层将维度从hid_dim*2降至hid_dim
            h_cat = torch.cat((hidden[layer][-2], hidden[layer][-1]), dim=1)
            c_cat = torch.cat((cell[layer][-2], cell[layer][-1]), dim=1)
            h_layer = torch.tanh(self.fc_hidden[layer](h_cat)).unsqueeze(0)
            c_layer = torch.tanh(self.fc_cell[layer](c_cat)).unsqueeze(0)

            final_hidden.append(h_layer)
            final_cell.append(c_layer)

        # 调整好维度为hid_dim的隐状态和细胞状态，可以传递给decoder
        hidden_concat = torch.cat(final_hidden, dim=0)
        cell_concat = torch.cat(final_cell, dim=0)

        return outputs, hidden_concat, cell_concat

In [5]:
# Attention

class Attention(nn.Module):
    # 通过一个两层的神经网络来计算attention
    def __init__(self, hid_dim):
        super(Attention, self).__init__()

        # 第一层输入维度是encoder每一时间步的输出隐状态(由于是双向，所以维度是hid_dim*2)和decoder当前时间步的输入隐状态(单向，维度是hid_dim)的拼接
        self.attn = nn.Linear(hid_dim*2 + hid_dim, hid_dim)
        # 输出一个代表注意力的logit值
        self.v = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs, mask):
        # 调整decoder当前时间步输入隐状态维度：[1, batch, hid_dim] -> [batch,1,hid_dim]
        hidden = hidden.permute(1,0,2)
        # 调整encoder各时间步输出隐状态的维度：[src_len,batch,hid_dim*2] -> [batch,src_len, hid_dim*2]
        encoder_outputs = encoder_outputs.permute(1,0,2)

        src_len = encoder_outputs.shape[1]
        # decoder中的token需要和encoder中所有token计算注意力，所有需要把decoder中token的状态复制多份，以便进行统一拼接
        # 因为decoder只有当前时间步输入的隐状态，复制到和encoder输出隐状态相同的src_len
        hidden = hidden.repeat(1, src_len, 1)   # [batch, src_len, hid_dim]

        # 拼接 decoder当前输入的隐状态和encoder各时间步输出的隐状态，然后经过一个线性层，tanh激活
        # [batch, src_len, hid_dim*3] -> [batch, src_len, hid_dim]
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        # 输出decoder当前token与encoder所有token的注意力值
        attention = self.v(energy).squeeze(2)   # [batch, src_len]
        # mask标志哪些位置是<pad>，这些位置上注意力值设为一个很大负数，softmax之后就是0
        attention = attention.masked_fill(mask == 0, -1e10)
        # 利用softmax将注意力的值归一化
        return torch.softmax(attention, dim=1)  # [batch, src_len]

In [6]:
# Decoder

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, attention, n_layers=3):
        super(Decoder, self).__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=PAD_ID)

        # 单向LSTM，输入维度为注意力加权后的encoder输出隐状态维度hid_dim*2加上输入token的embedding维度emb_dim
        self.rnn = nn.LSTM(hid_dim*2 + emb_dim, hid_dim, n_layers, bidirectional=False)

        # 最终分类头
        # 输入为hid_dim*3，输出为字典大小
        self.fc_out = nn.Linear(3*hid_dim, output_dim)

    def forward(self, input_token, hidden, cell, encoder_outputs, mask):
        # input_token: [batch]
        # 输入单个字符，增加一个维度
        input_token = input_token.unsqueeze(0)  # [1, batch]

        # 获取token的embedding
        embedded = self.embedding(input_token)  # [1, batch, emb_dim]

        # 获取当前步的输入隐状态
        last_hidden = hidden[-1].unsqueeze(0)

        # 当前步对所有encoder输出的注意力
        a = self.attention(last_hidden, encoder_outputs, mask)  # [batch, src_len]
        a = a.unsqueeze(1)  # [batch, 1, src_len]

        encoder_outputs = encoder_outputs.permute(1, 0, 2)  # [batch, src_len, enc_hid_dim*2]

        # 矩阵乘法获取注意力上下文向量
        weighted = torch.bmm(a, encoder_outputs)    # [batch, 1, enc_hid_dim*2]
        weighted = weighted.permute(1, 0, 2)    # [1, batch, enc_hid_dim*2]

        # 拼接输入token编码向量和注意力上下文向量
        lstm_input = torch.cat((embedded, weighted), dim=2) # [1, batch, enc_hid_dim*2+emb_dim]

        # 一次只执行lstm的一个时间步
        output, (hidden, cell) = self.rnn(lstm_input, (hidden, cell))   # output: [1, batch, hid_dim]

        # 移除第0维
        output = output.squeeze(0)
        weighted = weighted.squeeze(0)

        # 拼接输出和注意力上下文向量，进入分类头，计算分类logits
        prediction = self.fc_out(torch.cat((output, weighted), dim=1))  # [batch, output_dim]

        return prediction, hidden, cell, a.squeeze(1)   # attention weights for visualization

In [7]:
# Seq2Seq: 将encoder和decoder组装
# 输入英文token_id，一次性调用encoder，输出output、hidden、cell向量，然后逐步调用decoder

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, src_lens, trg):
        batch_size = trg.shape[1]
        max_len = trg.shape[0]
        vocab_size = self.decoder.output_dim

        outputs = torch.zeros(max_len, batch_size, vocab_size).to(self.device)

        # 调用encoder
        encode_outputs, hidden, cell = self.encoder(src, src_lens)

        input_token = trg[0]
        mask = (src !=PAD_ID).permute(1,0)

        # 逐步调用decoder
        for t in range(1, max_len):
            output, hidden, cell, _ = self.decoder(input_token, hidden, cell, encode_outputs, mask)
            outputs[t] = output
            input_token = trg[t]

        return outputs

In [8]:
# 训练

def train(model, iterator, optimizer, criterion):
    model.train()
    epoch_loss = 0
    step_loss = 0   # 用于累计每个step的loss
    step_count = 0  # 当前step计数器

    for i, (src, trg, src_len, _) in enumerate(iterator):
        src, trg = src.to(model.device), trg.to(model.device)
        optimizer.zero_grad()
        output = model(src, src_len, trg)
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)
        loss = criterion(output, trg)
        loss.backward()
        optimizer.step()

        # 更新损失统计
        step_loss += loss.item()
        epoch_loss += loss.item()
        step_count += 1

        if (i+1) % 100 == 0:
            avg_step_loss = step_loss / step_count
            print(f"Step {i+1}/{len(iterator)} | Loss: {avg_step_loss:.3f}")
            step_loss = 0   # 重置step损失
            step_count = 0  # 重置step计数器

    return epoch_loss / len(iterator)   # 返回整个epoch的平均loss

In [10]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataset = TranslationDataset(
        src_file=r"D:\agent\torch\data\en2cn\train_en_10000.txt",
        trg_file=r"D:\agent\torch\data\en2cn\train_zh_10000.txt",
        src_tokenizer=tokenize_en,
        trg_tokenizer=tokenize_cn,
    )
    loader = DataLoader(dataset, batch_size=32, collate_fn=TranslationDataset.collate_fn, shuffle=True)

    INPUT_DIM = sp_en.get_piece_size()  # 获取词表大小vocab_size
    OUTPUT_DIM = sp_cn.get_piece_size()
    ENC_EMB_DIM = 64
    DEC_EMB_DIM = 64
    HID_DIM = 64

    attention = Attention(HID_DIM).to(device)
    encoder = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM).to(device)
    decoder = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, attention).to(device)
    model = Seq2Seq(encoder, decoder, device).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

    N_EPOCHS = 5

    for epoch in range(N_EPOCHS):
        loss = train(model, loader, optimizer, criterion)
        print(f'Epoch {epoch + 1}/{N_EPOCHS} | Loss: {loss:.4f}')

    torch.save(
        model.state_dict(),
        r'C:\Users\61075\PycharmProjects\learning\torch\runs\translation\seq2seq_bpe_attention.pt'
    )

Step 100/313 | Loss: 7.181
Step 200/313 | Loss: 6.293
Step 300/313 | Loss: 6.072
Epoch 1/5 | Loss: 6.4942
Step 100/313 | Loss: 5.770
Step 200/313 | Loss: 5.757
Step 300/313 | Loss: 5.736
Epoch 2/5 | Loss: 5.7511
Step 100/313 | Loss: 5.559
Step 200/313 | Loss: 5.592
Step 300/313 | Loss: 5.556
Epoch 3/5 | Loss: 5.5663
Step 100/313 | Loss: 5.391
Step 200/313 | Loss: 5.381
Step 300/313 | Loss: 5.340
Epoch 4/5 | Loss: 5.3706
Step 100/313 | Loss: 5.162
Step 200/313 | Loss: 5.140
Step 300/313 | Loss: 5.189
Epoch 5/5 | Loss: 5.1620


模型评估

In [15]:
model = Seq2Seq(encoder, decoder, device).to(device)
model.load_state_dict(torch.load(r'C:\Users\61075\PycharmProjects\learning\torch\runs\translation\seq2seq_bpe_attention.pt', map_location=device))
model.eval()

def translate_sentence(sentence, max_len=100):
    ### greedy

    # tokenize and convert to IDs
    tokens = [BOS_ID] + sp_en.encode(sentence, out_type=int) + [EOS_ID]
    src_tensor = torch.LongTensor(tokens).unsqueeze(1).to(device)   # [src_len, 1]
    src_len = [len(tokens)]

    # encode
    with torch.no_grad():
        encoder_outputs, hidden, cell = encoder(src_tensor, src_len)

    # 第一个输入token，序列起始token：<bos>
    trg_indices = [BOS_ID]
    # 逐个token，循环调用decoder
    for _ in range(max_len):
        # 最新生成的token作为输入
        trg_tensor = torch.LongTensor([trg_indices[-1]]).to(device)
        with torch.no_grad():
            output, hidden, cell, _ = decoder(trg_tensor, hidden, cell, encoder_outputs,
                                              (src_tensor != PAD_ID).permute(1, 0))
        # 取预测概率最大的token作为输出
        pred_token = output.argmax(1).item()
        trg_indices.append(pred_token)
        if pred_token == EOS_ID:
            break

    # 将token_id解码为文字，并跳过<bos>和<eos>
    translated = sp_cn.decode(trg_indices[1:-1])
    return translated


if __name__ == '__main__':
    while True:
        src_sent = input("Enter English sentence (or 'quit' to exit): ")
        if src_sent.lower() in ['quit', 'exit']:
            break
        translation = translate_sentence(src_sent)
        print(f"Chinese Translation: {translation}\n")

Chinese Translation: 一年你



In [17]:
# BLEU指标评估
import sacrebleu

# 读取验证集的英文原文和中文原文(都只取前20个)
with open(r"D:\agent\torch\data\en2cn\valid_en.txt", "r", encoding="utf-8") as f:
    src_sentences = [line.strip() for _, line in zip(range(20), f)]
with open(r"D:\agent\torch\data\en2cn\valid_zh.txt", "r", encoding="utf-8") as f:
    ref_sentences = [line.strip() for _, line in zip(range(20), f)]

# 检查长度是否匹配
assert len(src_sentences) == len(ref_sentences), "源语言和参考翻译句子数不匹配"

# 生成翻译
hypotheses = []
for i, src in enumerate(src_sentences):
    print(f"translating {i+1}/{len(src_sentences)}...")
    translation = translate_sentence(src)
    print(ref_sentences[i], translation)
    hypotheses.append(translation.strip())  # 去掉首和尾的空格后添加

# 计算BLEU
bleu = sacrebleu.corpus_bleu(hypotheses, [ref_sentences], tokenize='zh')
print("\n========== BLEU Evaluation Result ==========")
print(f"BLEU Score: {bleu.score:.2f}")

translating 1/20...
你们觉得我们看起来够年轻溜进高中吗？ 一想到你的的的的的的的的的的?
translating 2/20...
嗨，亲爱的。你现在肯定忙着开会呢。 一小时所有。我。
translating 3/20...
因为你想在进养老院前娶妻生子。 一想到前在我就的的的的的。
translating 4/20...
我就一天24小时都得在她眼皮子底下。 一想到我我就的我就。
translating 5/20...
找条牢靠的链子或者别的什么固定住这些灯。 一座夫妇的的的的的。
translating 6/20...
为了不让别的父母经历我的遭遇。 一想到我我就我就我就我就。
translating 7/20...
我要去赴约会，必须学跳舞。现在就学。 一想到我,我我,我我,我我。
translating 8/20...
有时候我们信任的人替我们做了这样的选择。 一想到我我就的我就的我就。
translating 9/20...
好吧。那么，我想现在能做的有限。 一小时前,我,我,我。我。
translating 10/20...
我尊重这点，并且会不惜一切保护隐私不被侵犯。 一开始,是,后来后来,
translating 11/20...
太奇怪了。- 我们出去吧。 一小时前,我我。
translating 12/20...
所以在调查了有微量血迹的门把手之后， 一想到前,一一一手的,
translating 13/20...
如果我们找不到她，她就只有两周可活了。 一方面,我,我我,我我我,我我。
translating 14/20...
但是如果我们能摒弃前嫌 一方面,我,
translating 15/20...
如果你还想做球队经理的话你必须得做这个。 一想到我,我,我,我,我我。
translating 16/20...
好吗？你挥着你的小翅膀飞下来然后告诉他我不知道。 一把?一一一一一一手。
translating 17/20...
因为你要知道不管你在保护谁等你们一起进了号子 一想到我我我就的我就的的的的
translating 18/20...
你们还可以看到剧场台阶保存得十分完好； 一想到我我就的的的的的的的的
translating 19/20...
你得照顾他的孩子，我是说在干掉他之后？ 一把,一,一一,一?
tran